In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from typing import Callable
from PIL import Image

def extract_rgb_channels(img):
    """Extract RGB channels from the input image."""
    return img[:, :, 0], img[:, :, 1], img[:, :, 2]

def plot_mean_rgb_distribution(
    image_folder: str,
    output_folder: str,
    func: Callable,
    labels: list[str]
):
    """
    Iterate over JPG images, compute mean RGB scatter distributions, and save the plot.

    Args:
    ----
    image_folder: str
        Folder path containing JPG images.
    output_folder: str
        Folder to save output plots.
    func: Callable
        Function that extracts channels.
    labels: list of str
        Channel labels (e.g., ["Red", "Green", "Blue"])
    """
    red_vals = []
    green_vals = []
    blue_vals = []

    for fname in os.listdir(image_folder):
        if fname.lower().endswith(".jpg"):
            img_path = os.path.join(image_folder, fname)
            img = np.array(Image.open(img_path).convert("RGB"))
            r, g, b = func(img)

            # Downsample (10% for speed)
            mask = np.random.RandomState(seed=0).rand(*r.shape) < 0.1
            red_vals.append(r[mask])
            green_vals.append(g[mask])
            blue_vals.append(b[mask])

    # Flatten and concatenate all values
    red_all = np.concatenate(red_vals)
    green_all = np.concatenate(green_vals)
    blue_all = np.concatenate(blue_vals)

    # Normalize colors for scatter (approximation)
    rgb_colors = np.stack([red_all, green_all, blue_all], axis=1) / 255.0

    # Create figure
    fig = plt.figure(figsize=(16, 5))
    ax1 = fig.add_subplot(1, 3, 1)
    ax1.scatter(red_all, green_all, c=rgb_colors, s=1, alpha=0.1)
    ax1.set_xlabel(labels[0])
    ax1.set_ylabel(labels[1])
    ax1.set_title(f"{labels[0]} vs {labels[1]}")

    ax2 = fig.add_subplot(1, 3, 2)
    ax2.scatter(red_all, blue_all, c=rgb_colors, s=1, alpha=0.1)
    ax2.set_xlabel(labels[0])
    ax2.set_ylabel(labels[2])
    ax2.set_title(f"{labels[0]} vs {labels[2]}")

    ax3 = fig.add_subplot(1, 3, 3)
    ax3.scatter(green_all, blue_all, c=rgb_colors, s=1, alpha=0.1)
    ax3.set_xlabel(labels[1])
    ax3.set_ylabel(labels[2])
    ax3.set_title(f"{labels[1]} vs {labels[2]}")

    plt.tight_layout()

    os.makedirs(output_folder, exist_ok=True)
    output_path = os.path.join(output_folder, "mean_rgb_distribution.png")
    plt.savefig(output_path)
    plt.close()

# Example usage:
plot_mean_rgb_distribution(
    image_folder="../dataset_project_iapr2025/train",
    output_folder="RGB_channels",
    func=extract_rgb_channels,
    labels=["Red", "Green", "Blue"]
)